In [28]:
from toolbox_langchain import ToolboxClient
from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    ToolCall,
    ToolMessage,
)
from langchain_google_vertexai import ChatVertexAI
from langgraph.checkpoint.memory import MemorySaver
import vertexai
from google import genai
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [20]:
with ToolboxClient("http://127.0.0.1:5000") as client:
    tools = await client.aload_toolset("cymbal_air")
    print(tools)

[ToolboxTool(name='search_airports', description='Use this tool to list all airports matching search criteria.\nTakes at least one of country, city, name, or all and returns all matching airports.\nThe agent can decide to return the results directly to the user.\n\nArgs:\n    city (Optional): City\n    country (Optional): Country\n    name (Optional): Airport name', args_schema=<class 'toolbox_core.utils.search_airports'>), ToolboxTool(name='list_flights', description='Use this tool to list flights information matching search criteria.\nTakes an arrival airport, a departure airport, or both, filters by date and returns all matching flights.\nIf 3-letter iata code is not provided for departure_airport or arrival_airport, use `search_airports` tool to get iata code information.\nDo NOT guess a date, ask user for date input if it is not given. Date must be in the following format: YYYY-MM-DD.\nThe agent can decide to return the results directly to the user.\n\nArgs:\n    arrival_airport (

In [21]:
len(tools)

7

In [22]:
async with ToolboxClient("http://127.0.0.1:5000") as client:
    search_airports = await client.aload_tool("search_airports")

    result = await search_airports.ainvoke({
        "country": "Indonesia",
        "city": "Jakarta",
        "name": ""
    })

    print(result)

[{"id":3088,"iata":"CGK","name":"Soekarno-Hatta International Airport","city":"Jakarta","country":"Indonesia"},{"id":3700,"iata":"PCB","name":"Pondok Cabe Air Base","city":"Jakarta","country":"Indonesia"},{"id":4857,"iata":"HLP","name":"Halim Perdanakusuma International Airport","city":"Jakarta","country":"Indonesia"}]


In [23]:
async with ToolboxClient("http://127.0.0.1:5000") as client:
    list_flights = await client.aload_tool("list_flights")

    result = await list_flights.ainvoke({
        "departure_airport": "SFO",
        "arrival_airport": "DEN",
        "date": "2025-01-01"
    })

    print(result)

[{"id":0,"airline":"UA","flight_number":"1532","departure_airport":"SFO","arrival_airport":"DEN","departure_time":"2025-01-01T05:50:00Z","arrival_time":"2025-01-01T09:23:00Z","departure_gate":"E49","arrival_gate":"D6"},{"id":59,"airline":"UA","flight_number":"720","departure_airport":"SFO","arrival_airport":"DEN","departure_time":"2025-01-01T12:46:00Z","arrival_time":"2025-01-01T16:19:00Z","departure_gate":"C41","arrival_gate":"E39"},{"id":84,"airline":"UA","flight_number":"1733","departure_airport":"SFO","arrival_airport":"DEN","departure_time":"2025-01-01T14:38:00Z","arrival_time":"2025-01-01T18:35:00Z","departure_gate":"B27","arrival_gate":"E32"},{"id":139,"airline":"F9","flight_number":"664","departure_airport":"SFO","arrival_airport":"DEN","departure_time":"2025-01-01T19:31:00Z","arrival_time":"2025-01-01T23:13:00Z","departure_gate":"E11","arrival_gate":"E12"},{"id":142,"airline":"UA","flight_number":"1243","departure_airport":"SFO","arrival_airport":"DEN","departure_time":"2025-0

In [24]:
async with ToolboxClient("http://127.0.0.1:5000") as client:
    search_flight = await client.aload_tool("search_flights_by_number")

    result = await search_flight.ainvoke({
        "airline": "UA",
        "flight_number": "1532"
    })

    print(result)

[{"id":0,"airline":"UA","flight_number":"1532","departure_airport":"SFO","arrival_airport":"DEN","departure_time":"2025-01-01T05:50:00Z","arrival_time":"2025-01-01T09:23:00Z","departure_gate":"E49","arrival_gate":"D6"},{"id":789,"airline":"UA","flight_number":"1532","departure_airport":"SFO","arrival_airport":"DEN","departure_time":"2025-01-05T05:28:00Z","arrival_time":"2025-01-05T08:55:00Z","departure_gate":"A22","arrival_gate":"C47"},{"id":1213,"airline":"UA","flight_number":"1532","departure_airport":"IAD","arrival_airport":"SFO","departure_time":"2025-01-07T08:34:00Z","arrival_time":"2025-01-07T11:40:00Z","departure_gate":"E16","arrival_gate":"D18"},{"id":40089,"airline":"UA","flight_number":"1532","departure_airport":"SNA","arrival_airport":"SFO","departure_time":"2025-07-26T16:47:00Z","arrival_time":"2025-07-26T17:57:00Z","departure_gate":"E22","arrival_gate":"B44"}]


In [38]:
async with ToolboxClient("http://127.0.0.1:5000") as client:
    search_amenities = await client.aload_tool("search_amenities")
    
    client = genai.Client()
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents="Cofee"
    )    
    embedding_str = str(result.embeddings[0].values)
    
    amenities = await search_amenities.ainvoke({
        "query_embedding": embedding_str
    })
    print(amenities)

[{"name":"Hawaii Coffee Store","description":"Serving authentic hawaiian coffee.","location":"Near Gate E2","terminal":"Terminal 3","category":"restaurant","hour":"Daily 7:00 am - 11:00 pm"},{"name":"Starbucks Coffee","description":"International coffeehouse chain serving hot and cold beverages, pastries, and sandwiches.","location":"Near Gate C18","terminal":"Terminal 1","category":"restaurant","hour":"Daily 5:00 am - 10:00 pm"},{"name":"Starbucks Coffee","description":"International coffeehouse chain serving hot and cold beverages, pastries, and sandwiches.","location":"Pre-Security","terminal":"All Terminals","category":"restaurant","hour":"Daily 5:00 am - 10:00 pm"},{"name":"Local Coffee Roaster","description":"Featuring specialty coffee from local roasters","location":"Near Gate D6","terminal":"Terminal 3","category":"restaurant","hour":"Daily 7:00 am - 8:00 pm"},{"name":"Coffee Shop 732","description":"Serving American cuisine.","location":"Near Gate B12","terminal":"Terminal 3",

In [29]:
async with ToolboxClient("http://127.0.0.1:5000") as client:
    search_policies = await client.aload_tool("search_policies")
    
    client = genai.Client()
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents="baggage policy"
    )    
    embedding_str = str(result.embeddings[0].values)
    
    policies = await search_policies.ainvoke({
        "query_embedding": embedding_str
    })
    print(policies)

[{"content":"Carry-on Baggage: Passengers are allowed one carry-on bag and one personal item. These items must meet size and weight restrictions.\nLiability: Cymbal Air assumes limited liability for lost, damaged, or delayed baggage. Passengers are encouraged to purchase travel insurance for additional protection."},{"content":"## Baggage\nChecked Baggage: Economy passengers are allowed 2 checked bags. Business class and First class passengers are allowed 4 checked bags. Additional baggage will cost $70 and a $30 fee applies for all checked bags over 50 lbs. Cymbal Air cannot accept checked bags over 100 lbs. We only accept checked bags up to 115 inches in total dimensions (length + width + height), and oversized baggage will cost $30. Checked bags above 160 inches in total dimensions will not be accepted."},{"content":"Traveling with Pets: Pets may be allowed in the cabin or as checked baggage depending on size and breed. Fees and restrictions apply. Please consult our website."},{"co

In [31]:
policies

'[{"content":"Carry-on Baggage: Passengers are allowed one carry-on bag and one personal item. These items must meet size and weight restrictions.\\nLiability: Cymbal Air assumes limited liability for lost, damaged, or delayed baggage. Passengers are encouraged to purchase travel insurance for additional protection."},{"content":"## Baggage\\nChecked Baggage: Economy passengers are allowed 2 checked bags. Business class and First class passengers are allowed 4 checked bags. Additional baggage will cost $70 and a $30 fee applies for all checked bags over 50 lbs. Cymbal Air cannot accept checked bags over 100 lbs. We only accept checked bags up to 115 inches in total dimensions (length + width + height), and oversized baggage will cost $30. Checked bags above 160 inches in total dimensions will not be accepted."},{"content":"Traveling with Pets: Pets may be allowed in the cabin or as checked baggage depending on size and breed. Fees and restrictions apply. Please consult our website."},{

In [32]:
import json

In [33]:
data = json.loads(policies)

In [35]:
len(data)

5

In [36]:
data[0]

{'content': 'Carry-on Baggage: Passengers are allowed one carry-on bag and one personal item. These items must meet size and weight restrictions.\nLiability: Cymbal Air assumes limited liability for lost, damaged, or delayed baggage. Passengers are encouraged to purchase travel insurance for additional protection.'}

In [37]:
data[1]

{'content': '## Baggage\nChecked Baggage: Economy passengers are allowed 2 checked bags. Business class and First class passengers are allowed 4 checked bags. Additional baggage will cost $70 and a $30 fee applies for all checked bags over 50 lbs. Cymbal Air cannot accept checked bags over 100 lbs. We only accept checked bags up to 115 inches in total dimensions (length + width + height), and oversized baggage will cost $30. Checked bags above 160 inches in total dimensions will not be accepted.'}